In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product

from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product

from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
#rutas
SPLIT_PATH_ADJ5 = "/content/drive/MyDrive/tesis/split_1.npz"

LABEL_ORDER_ADJ5 = [1, 2, 3, 4, 5]
CLASS_NAMES_ADJ5 = ["Hepatocito", "Estrellada", "Kupffer", "Endotelial", "Otras"]

RANDOM_STATE_ADJ5 = 42

# carpeta donde se guarda todo
OUTPUT_DIR_ADJ5 = "/content/drive/MyDrive/tesis/mlp_adjusted_5classes_pipeline_pca_macro"
os.makedirs(OUTPUT_DIR_ADJ5, exist_ok=True)

# output
RESULTS_CSV_PATH_ADJ5     = os.path.join(OUTPUT_DIR_ADJ5, "mlp_adjusted_5classes_results.csv")
BEST_PARAMS_TXT_PATH_ADJ5 = os.path.join(OUTPUT_DIR_ADJ5, "best_params_adjusted_5classes.txt")
MODEL_PATH_ADJ5           = os.path.join(OUTPUT_DIR_ADJ5, "mlp_adjusted_5classes_pipeline.pkl")

METRICS_CSV_PATH_ADJ5     = os.path.join(OUTPUT_DIR_ADJ5, "metrics_summary_adjusted_5classes.csv")
METRICS_TXT_PATH_ADJ5     = os.path.join(OUTPUT_DIR_ADJ5, "metrics_report_adjusted_5classes.txt")
METRICS_PNG_PATH_ADJ5     = os.path.join(OUTPUT_DIR_ADJ5, "metrics_summary_adjusted_5classes.png")

CM_VAL_PATH_ADJ5          = os.path.join(OUTPUT_DIR_ADJ5, "confusion_matrix_validacion_adjusted_5classes.png")
CM_TEST_PATH_ADJ5         = os.path.join(OUTPUT_DIR_ADJ5, "confusion_matrix_test_adjusted_5classes.png")
LOSS_CURVE_PATH_ADJ5      = os.path.join(OUTPUT_DIR_ADJ5, "loss_curve_adjusted_5classes.png")

print("Carpeta de salida:", OUTPUT_DIR_ADJ5)

In [ ]:
data_adj5 = np.load(SPLIT_PATH_ADJ5)

X_train_adj5 = data_adj5["X_train"]
y_train_adj5 = data_adj5["y_train"]

X_val_adj5 = data_adj5["X_val"]
y_val_adj5 = data_adj5["y_val"]

X_test_adj5 = data_adj5["X_test"]
y_test_adj5 = data_adj5["y_test"]

print("Shapes AJUSTADA 5 CLASES:")
print("X_train_adj5:", X_train_adj5.shape, "| y_train_adj5:", y_train_adj5.shape)
print("X_val_adj5:  ", X_val_adj5.shape,   "| y_val_adj5:  ", y_val_adj5.shape)
print("X_test_adj5: ", X_test_adj5.shape,  "| y_test_adj5: ", y_test_adj5.shape)

print("\nClases train:", np.unique(y_train_adj5))
print("Clases val:  ", np.unique(y_val_adj5))
print("Clases test: ", np.unique(y_test_adj5))

def print_class_counts_adj5(y, split_name):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\nConteo por clase - {split_name}")
    for u, c in zip(unique, counts):
        print(f"Clase {u}: {c}")

print_class_counts_adj5(y_train_adj5, "Train AJUSTADA")
print_class_counts_adj5(y_val_adj5, "Validación AJUSTADA")
print_class_counts_adj5(y_test_adj5, "Test AJUSTADA")

In [ ]:
#funciones definidas
def evaluate_model_adj5(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)

    precision_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall_w    = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_w        = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    precision_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_m    = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_m        = f1_score(y_true, y_pred, average="macro", zero_division=0)

    bal_acc = balanced_accuracy_score(y_true, y_pred)

    report_txt = classification_report(
        y_true,
        y_pred,
        labels=LABEL_ORDER_ADJ5,
        target_names=CLASS_NAMES_ADJ5,
        zero_division=0
    )

    print(f"\n{'='*60}")
    print(f"RESULTADOS - {split_name}")
    print(f"{'='*60}")
    print(f"Accuracy           : {acc:.4f}")
    print(f"Precision weighted : {precision_w:.4f}")
    print(f"Recall weighted    : {recall_w:.4f}")
    print(f"F1 weighted        : {f1_w:.4f}")
    print(f"Precision macro    : {precision_m:.4f}")
    print(f"Recall macro       : {recall_m:.4f}")
    print(f"F1 macro           : {f1_m:.4f}")
    print(f"Balanced Accuracy  : {bal_acc:.4f}")

    print("\nClassification Report:")
    print(report_txt)

    return {
        "split": split_name,
        "accuracy": acc,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "precision_macro": precision_m,
        "recall_macro": recall_m,
        "f1_macro": f1_m,
        "balanced_accuracy": bal_acc
    }, report_txt


def save_confusion_matrix_adj5(y_true, y_pred, split_name, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER_ADJ5)

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES_ADJ5
    )
    disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45, colorbar=False)
    plt.title(f"Matriz de confusión - {split_name}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Guardado: {save_path}")


def save_loss_curve_adj5(mlp_model, save_path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(mlp_model.loss_curve_)
    ax.set_title("Curva de pérdida - MLP AJUSTADA")
    ax.set_xlabel("Época / Iteración")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Guardado: {save_path}")


def save_metrics_table_png_adj5(metrics_df, save_path):
    df_show = metrics_df.copy().round(4)

    fig, ax = plt.subplots(figsize=(12, 2 + 0.5 * len(df_show)))
    ax.axis("off")

    table = ax.table(
        cellText=df_show.values,
        colLabels=df_show.columns,
        loc="center",
        cellLoc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    plt.title("Resumen de métricas - AJUSTADA 5 CLASES", pad=20)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Guardado: {save_path}")

In [ ]:
param_grid_adj5 = {
    "pca__n_components": [0.95, 256],
    "pca__whiten": [False],
    "mlp__hidden_layer_sizes": [
        (256, 128, 64),
        (512, 256),
        (256, 256, 128),
        (512, 256, 128)
    ],
    "mlp__alpha": [1e-4, 5e-4, 1e-3],
    "mlp__learning_rate_init": [1e-3, 5e-4],
    "mlp__batch_size": [64, 128]
}

all_combinations_adj5 = list(product(
    param_grid_adj5["pca__n_components"],
    param_grid_adj5["pca__whiten"],
    param_grid_adj5["mlp__hidden_layer_sizes"],
    param_grid_adj5["mlp__alpha"],
    param_grid_adj5["mlp__learning_rate_init"],
    param_grid_adj5["mlp__batch_size"]
))

print(f"Total de combinaciones AJUSTADA: {len(all_combinations_adj5)}")

results_adj5 = []
best_score_adj5 = -1
best_params_adj5 = None
best_model_adj5 = None

for i, (n_components, whiten, hidden_layer_sizes, alpha, learning_rate_init, batch_size) in enumerate(all_combinations_adj5, start=1):
    print(f"\n[{i}/{len(all_combinations_adj5)}] Probando AJUSTADA:")
    print(f"pca__n_components={n_components}, pca__whiten={whiten}, "
          f"hidden_layer_sizes={hidden_layer_sizes}, alpha={alpha}, "
          f"learning_rate_init={learning_rate_init}, batch_size={batch_size}")

    pipe_adj5 = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=n_components,
            whiten=whiten,
            random_state=RANDOM_STATE_ADJ5
        )),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation="relu",
            solver="adam",
            alpha=alpha,
            batch_size=batch_size,
            learning_rate_init=learning_rate_init,
            max_iter=500,
            shuffle=True,
            random_state=RANDOM_STATE_ADJ5,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            verbose=False
        ))
    ])

    pipe_adj5.fit(X_train_adj5, y_train_adj5)
    y_val_pred_adj5 = pipe_adj5.predict(X_val_adj5)

    acc_val_adj5 = accuracy_score(y_val_adj5, y_val_pred_adj5)
    f1_w_val_adj5 = f1_score(y_val_adj5, y_val_pred_adj5, average="weighted", zero_division=0)
    f1_m_val_adj5 = f1_score(y_val_adj5, y_val_pred_adj5, average="macro", zero_division=0)
    bal_acc_val_adj5 = balanced_accuracy_score(y_val_adj5, y_val_pred_adj5)

    results_adj5.append({
        "pca__n_components": n_components,
        "pca__whiten": whiten,
        "hidden_layer_sizes": str(hidden_layer_sizes),
        "alpha": alpha,
        "learning_rate_init": learning_rate_init,
        "batch_size": batch_size,
        "accuracy_val": acc_val_adj5,
        "f1_weighted_val": f1_w_val_adj5,
        "f1_macro_val": f1_m_val_adj5,
        "balanced_acc_val": bal_acc_val_adj5,
        "n_iter_": pipe_adj5.named_steps["mlp"].n_iter_
    })

    print(f"Val Accuracy    : {acc_val_adj5:.4f}")
    print(f"Val F1 weighted : {f1_w_val_adj5:.4f}")
    print(f"Val F1 macro    : {f1_m_val_adj5:.4f}")
    print(f"Val Bal Accuracy: {bal_acc_val_adj5:.4f}")
    print(f"n_iter_         : {pipe_adj5.named_steps['mlp'].n_iter_}")

    # Selección por F1 macro
    if f1_m_val_adj5 > best_score_adj5:
        best_score_adj5 = f1_m_val_adj5
        best_params_adj5 = {
            "pca__n_components": n_components,
            "pca__whiten": whiten,
            "hidden_layer_sizes": hidden_layer_sizes,
            "alpha": alpha,
            "learning_rate_init": learning_rate_init,
            "batch_size": batch_size
        }
        best_model_adj5 = pipe_adj5

results_df_adj5 = pd.DataFrame(results_adj5).sort_values(
    by="f1_macro_val", ascending=False
).reset_index(drop=True)

print("\nTop 10 configuraciones AJUSTADA:")
print(results_df_adj5.head(10))

print("\nMejores hiperparámetros AJUSTADA:")
print(best_params_adj5)
print(f"Mejor F1 macro en validación: {best_score_adj5:.4f}")

results_df_adj5.to_csv(RESULTS_CSV_PATH_ADJ5, index=False)
print("Resultados de búsqueda guardados en:")
print(RESULTS_CSV_PATH_ADJ5)

In [ ]:
final_model_adj5 = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(
        n_components=best_params_adj5["pca__n_components"],
        whiten=best_params_adj5["pca__whiten"],
        random_state=RANDOM_STATE_ADJ5
    )),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=best_params_adj5["hidden_layer_sizes"],
        activation="relu",
        solver="adam",
        alpha=best_params_adj5["alpha"],
        batch_size=best_params_adj5["batch_size"],
        learning_rate_init=best_params_adj5["learning_rate_init"],
        max_iter=500,
        shuffle=True,
        random_state=RANDOM_STATE_ADJ5,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        verbose=False
    ))
])

final_model_adj5.fit(X_train_adj5, y_train_adj5)

print("Modelo final AJUSTADO entrenado con Scaler + PCA + MLP.")
print("Iteraciones usadas:", final_model_adj5.named_steps["mlp"].n_iter_)

y_train_pred_adj5 = final_model_adj5.predict(X_train_adj5)
y_val_pred_adj5   = final_model_adj5.predict(X_val_adj5)
y_test_pred_adj5  = final_model_adj5.predict(X_test_adj5)

train_metrics_adj5, train_report_adj5 = evaluate_model_adj5(y_train_adj5, y_train_pred_adj5, "Train AJUSTADA")
val_metrics_adj5, val_report_adj5     = evaluate_model_adj5(y_val_adj5, y_val_pred_adj5, "Validación AJUSTADA")
test_metrics_adj5, test_report_adj5   = evaluate_model_adj5(y_test_adj5, y_test_pred_adj5, "Test AJUSTADA")

metrics_df_adj5 = pd.DataFrame([
    train_metrics_adj5,
    val_metrics_adj5,
    test_metrics_adj5
])

print("\nResumen de métricas AJUSTADA:")
print(metrics_df_adj5.round(4))

In [ ]:
save_confusion_matrix_adj5(y_val_adj5, y_val_pred_adj5, "Validación AJUSTADA", CM_VAL_PATH_ADJ5)
save_confusion_matrix_adj5(y_test_adj5, y_test_pred_adj5, "Test AJUSTADA", CM_TEST_PATH_ADJ5)

save_loss_curve_adj5(final_model_adj5.named_steps["mlp"], LOSS_CURVE_PATH_ADJ5)

metrics_df_adj5.to_csv(METRICS_CSV_PATH_ADJ5, index=False)
save_metrics_table_png_adj5(metrics_df_adj5, METRICS_PNG_PATH_ADJ5)

with open(BEST_PARAMS_TXT_PATH_ADJ5, "w", encoding="utf-8") as f:
    f.write("MEJORES HIPERPARÁMETROS - AJUSTADA 5 CLASES\n")
    f.write("=" * 80 + "\n")
    for k, v in best_params_adj5.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nMejor F1 macro en validación: {best_score_adj5:.6f}\n")

with open(METRICS_TXT_PATH_ADJ5, "w", encoding="utf-8") as f:
    f.write("REPORTE COMPLETO - AJUSTADA 5 CLASES\n")
    f.write("=" * 80 + "\n\n")

    f.write("Shapes\n")
    f.write("-" * 80 + "\n")
    f.write(f"X_train_adj5: {X_train_adj5.shape} | y_train_adj5: {y_train_adj5.shape}\n")
    f.write(f"X_val_adj5:   {X_val_adj5.shape} | y_val_adj5:   {y_val_adj5.shape}\n")
    f.write(f"X_test_adj5:  {X_test_adj5.shape} | y_test_adj5:  {y_test_adj5.shape}\n\n")

    f.write("Mejores hiperparámetros\n")
    f.write("-" * 80 + "\n")
    for k, v in best_params_adj5.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nMejor F1 macro en validación: {best_score_adj5:.6f}\n\n")

    f.write("Resumen numérico de métricas\n")
    f.write("-" * 80 + "\n")
    f.write(metrics_df_adj5.to_string(index=False))
    f.write("\n\n")

    f.write("Classification Report - Train AJUSTADA\n")
    f.write("-" * 80 + "\n")
    f.write(train_report_adj5)
    f.write("\n\n")

    f.write("Classification Report - Validación AJUSTADA\n")
    f.write("-" * 80 + "\n")
    f.write(val_report_adj5)
    f.write("\n\n")

    f.write("Classification Report - Test AJUSTADA\n")
    f.write("-" * 80 + "\n")
    f.write(test_report_adj5)
    f.write("\n\n")

joblib.dump(final_model_adj5, MODEL_PATH_ADJ5)

print("Archivos AJUSTADA guardados correctamente:")
print(RESULTS_CSV_PATH_ADJ5)
print(BEST_PARAMS_TXT_PATH_ADJ5)
print(MODEL_PATH_ADJ5)
print(METRICS_CSV_PATH_ADJ5)
print(METRICS_TXT_PATH_ADJ5)
print(METRICS_PNG_PATH_ADJ5)
print(CM_VAL_PATH_ADJ5)
print(CM_TEST_PATH_ADJ5)
print(LOSS_CURVE_PATH_ADJ5)

In [ ]:
print("Mejores hiperparámetros finales AJUSTADA:")
print(best_params_adj5)

print("\nTop 10 combinaciones por F1 macro:")
print(results_df_adj5.head(10))